In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from source.version0.model import ResnetModel
from source.version0.tta import ScoreDataset, scoreModel

In [3]:
def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(2017)

In [4]:
def score(idx):
    fold = idx if idx <=5 else idx - 5
    data = pd.read_csv('../../data/sample_submission.csv')
    driver = data[['recording_id']].copy()
    data = ScoreDataset(data)
    data = DataLoader(data, batch_size=1, shuffle=False, num_workers=7)
    model = ResnetModel()
    model = model.to('cuda:0')
    weights = torch.load('../../model/version0/model_{}.pt'.format(fold))
    weights = weights['model_state_dict']
    model.load_state_dict(weights)
    scores = scoreModel(model, data)
    scores = pd.DataFrame(scores)
    scores.columns = ['s' + str(x) for x in range(24)]
    scores = driver.join(scores)
    scores.to_csv('../../model/tta/score_{}.csv'.format(idx), index=False)
    return None

In [5]:
score(1)

In [6]:
score(2)

In [7]:
score(3)

In [8]:
score(4)

In [9]:
score(5)

In [10]:
score(6)

In [11]:
score(7)

In [12]:
score(8)

In [13]:
score(9)

In [14]:
score(10)

In [15]:
score1 = pd.read_csv('../../model/tta/score_1.csv')
score2 = pd.read_csv('../../model/tta/score_2.csv')
score3 = pd.read_csv('../../model/tta/score_3.csv')
score4 = pd.read_csv('../../model/tta/score_4.csv')
score5 = pd.read_csv('../../model/tta/score_5.csv')
score6 = pd.read_csv('../../model/tta/score_6.csv')
score7 = pd.read_csv('../../model/tta/score_7.csv')
score8 = pd.read_csv('../../model/tta/score_8.csv')
score9 = pd.read_csv('../../model/tta/score_9.csv')
score10 = pd.read_csv('../../model/tta/score_10.csv')

In [16]:
score = score1.append(score2).append(score3).append(score4).append(score5)
score = score.append(score6).append(score7).append(score8).append(score9).append(score10)
score = score.groupby('recording_id').mean().reset_index()

In [18]:
score.to_csv('../../score/version0_tta.csv', index=False)